# 이미지 생성 및 평가와 모델 학습 — 자기주도 실습

---

> **📌 이 노트북의 사용 방법**
>
> 이 노트북은 셀을 하나씩 읽고 실행하며 학습하는 자료로 보완되었습니다.
> - 각 코드 셀의 **주석을 반드시 먼저 읽은 후** 실행할 것
> - 각 Step이 끝날 때마다 **"관찰 포인트"**를 확인하여 결과를 스스로 해석할 것

> **💡 이 실습의 위치: CNN → 전이학습 → 파운데이션 모델**
>
> 3-1에서는 사전학습된 CNN(ResNet)을 **가져다 재활용**하는 전이학습을 배웠다.
> 이번 3-2에서는 한 단계 더 나아간다:
>
> ```
> 3-1 전이학습: "학습된 CNN을 가져다 쓴다"
>  → 3-2 파운데이션 모델: "텍스트로 이미지를 만들고, 평가하고, 학습 데이터로까지 활용한다"
> ```

## 학습 개요

### 학습 주제
- **텍스트-투-이미지 생성**: 디퓨전 기반 생성 모델(SANA)을 활용한 프롬프트 기반 이미지 생성
- **멀티모달 이미지 평가**: CLIP 모델을 활용한 이미지-텍스트 유사도 측정 및 CNN과의 비교
- **전이학습과 합성 데이터**: 생성 이미지를 활용한 ResNet-18 전이학습(Linear Probing) 파이프라인

### 학습 목표
- 디퓨전 모델의 이미지 생성 원리를 이해하고, **SANA로 텍스트 기반 이미지를 생성**할 수 있다
- CLIP 모델을 활용하여 **이미지-텍스트 간 유사도를 측정**하고 결과를 해석할 수 있다
- 동일 이미지에 대해 **CLIP과 ResNet-50의 분류 결과를 비교**하고 차이를 설명할 수 있다
- 합성 데이터셋을 구성하고 **ResNet-18에 대해 Linear Probing을 구현**할 수 있다

### 실습 구성 (4 Step)

| Step | 내용 | 핵심 개념 | 교안 연결 | 소요 시간 |
|:---:|------|----------|:---:|:---:|
| 1 | **SANA로 이미지 생성** | Diffusion, LDM, 프롬프트 | 슬라이드 14~20, 34~35 | 약 20분 |
| 2 | **CLIP으로 생성 이미지 평가** | CLIP Score, 코사인 유사도 | 슬라이드 24~26 | 약 10분 |
| 3 | **ResNet-50 vs CLIP 비교** | CNN vs 멀티모달 | 슬라이드 5 | 약 10분 |
| 4 | **합성 데이터로 전이학습** | Linear Probing, 합성 데이터 | 슬라이드 36 + 3-1 복습 | 약 20분 |


## 실습 구성

1. **학습 방향**


- **Required Package**
  ```
  python==3.11
  numpy>=2.0.0
  torch>=2.1.0
  torchvision>=0.16.0
  transformers==4.57.1
  diffusers>=0.30.0
  accelerate>=0.30.0
  Pillow>=10.0.0
  hf_transfer>=0.1.9
  ```

- **Step 요약 (총 60분)**
  - **Step 1 (20분)**: 텍스트-투-이미지 생성 — SANA 모델을 로드하고, 프롬프트를 설계하여 수채화 스타일의 이미지를 생성한다.
  - **Step 2 (10분)**: CLIP 모델을 사용한 이미지 평가 — CLIP 모델로 생성 이미지와 텍스트 레이블 간 의미적 유사도를 측정하고 결과를 해석한다.
  - **Step 3 (10분)**: ResNet-50 기반 분류 비교 — 동일 이미지를 ResNet-50으로 분류하고, CLIP 결과와 비교하여 모델 특성 차이를 이해한다.
  - **Step 4 (20분)**: 생성 데이터로 ResNet-18 전이학습 — 합성 데이터셋을 구성하고, ResNet-18의 출력층을 교체하여 리니어 프로빙을 수행한다.

2. **문제 설명**
  - **문제 개요**: 이 실습은 **이미지 파운데이션 모델의 활용과 비교**를 위해 설계되었습니다. 학습자는 **디퓨전 모델, CLIP, ResNet 등 다양한 사전학습 모델의 개념**을 코드로 확인하고, 최종적으로 **텍스트 기반 이미지 생성 → 멀티모달 평가 → CNN 비교 → 전이학습**으로 이어지는 파이프라인을 구현할 수 있어야 합니다.

  - **요구사항 요약**
    - **이미지 생성**: SANA 파이프라인을 로드하고 프롬프트 기반 이미지 생성
    - **CLIP 평가**: 생성 이미지와 텍스트 레이블 간 유사도 측정 및 해석
    - **CNN 비교**: ResNet-50 분류 결과와 CLIP 결과 비교 분석
    - **전이학습**: 합성 데이터셋으로 ResNet-18 리니어 프로빙 구현

3. **학습 문제: Step–TODO 구체 설명**
  - **Step 1 — 텍스트-투-이미지 생성**
    - **TODO 1**: SANA 파이프라인 로드 및 이미지 생성 *(연결 핵심개념: 디퓨전 모델 / 프롬프트 엔지니어링)*
    - **1줄 요약**: SANA 모델을 로드하고 positive 프롬프트로 수채화 여우 이미지를 생성한다.

  - **Step 2 — CLIP 모델을 사용한 이미지 평가**
    - **TODO 2**: CLIP 이미지-텍스트 유사도 계산 *(연결 핵심개념: CLIP / 멀티모달 평가)*
    - **1줄 요약**: CLIP 모델로 생성 이미지와 여러 텍스트 레이블 간 유사도를 계산한다.

  - **Step 3 — ResNet-50 기반 분류 비교**
    - **TODO 3**: ResNet-50으로 이미지 분류 *(연결 핵심개념: CNN / ImageNet 사전학습)*
    - **1줄 요약**: ImageNet 사전학습된 ResNet-50으로 동일 이미지를 분류하고 CLIP 결과와 비교한다.

  - **Step 4 — 생성 데이터로 ResNet-18 전이학습**
    - **TODO 4**: ResNet-18 전이학습(리니어 프로빙) 구현 *(연결 핵심개념: 전이학습 / 합성 데이터)*
    - **1줄 요약**: 합성 데이터셋으로 ResNet-18의 fc층을 교체하고 리니어 프로빙을 수행한다.


## Import & Install

실습에 필요한 라이브러리를 설치하고 불러온다.

> **💡 이번 실습에서 새로 등장하는 라이브러리**
>
> | 라이브러리 | 역할 | 3-1에서 사용? |
> |------|------|:---:|
> | `diffusers` | SANA 등 디퓨전 모델 파이프라인 | ❌ 새로 추가 |
> | `transformers` | CLIP, ViT 등 HuggingFace 모델 | ✅ (ViT에서 사용) |
> | `torchvision` | ResNet, 이미지 전처리 | ✅ (ResNet-18에서 사용) |
> | `accelerate` | GPU 메모리 최적화 (CPU offload 등) | ❌ 새로 추가 |
> | `PIL` | 이미지 파일 열기/저장 | ✅ |


In [ ]:
# 공통 실습 환경 설치 (최초 1회 실행)
!pip install -q \
    "numpy>=2.0.0" \
    "torch>=2.1.0" \
    "torchvision>=0.16.0" \
    "transformers==4.57.1" \
    "diffusers>=0.30.0" \
    "accelerate>=0.30.0" \
    "Pillow>=10.0.0" \
    "hf_transfer>=0.1.9"


In [ ]:
# ========== 라이브러리 임포트 ==========
import os                          # 파일/폴더 경로 관리
import numpy as np                 # 수치 연산
import random                      # 파이썬 기본 난수
import torch                       # PyTorch 핵심
import torch.nn as nn              # 모델(레이어) 정의 도구
import torch.optim as optim        # 옵티마이저 (SGD, Adam 등)
from PIL import Image              # 이미지 파일 열기/저장/표시
from torchvision import transforms, models  # 이미지 전처리 + 사전학습 모델
from torch.utils.data import DataLoader      # 미니배치 데이터 공급
from torchvision.datasets import ImageFolder  # 폴더 구조 → 데이터셋 자동 변환

# ========== 시드 고정 ==========
# 왜 시드를 고정하는가?
# → 딥러닝에서는 가중치 초기화, 데이터 셔플, 노이즈 생성 등에 난수가 사용된다.
#   시드를 고정하면 매번 동일한 결과를 얻을 수 있어 실험 재현이 가능하다.
torch.manual_seed(42)
torch.cuda.manual_seed(42)
np.random.seed(42)
random.seed(42)

# CuDNN의 비결정적 알고리즘을 비활성화 (GPU 연산 재현성 확보)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ========== 장치 설정 ==========
# GPU가 있으면 GPU, 없으면 CPU 사용
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 중인 장치: {device}')


## Step 1: 텍스트-투-이미지 생성

### 디퓨전 모델(Diffusion Model)이란?

> **💡 교안 복습 (슬라이드 14~16)**
>
> 디퓨전 모델은 **노이즈가 가득한 랜덤 이미지에서 시작**하여, 텍스트 조건에 맞게
> **점진적으로 노이즈를 제거**해가며 최종 이미지를 생성하는 모델이다.
>
> ```
> [순수 노이즈] → 텍스트를 참고하며 노이즈 조금 제거 → ... → [선명한 이미지]
> ```
>
> GAN처럼 두 모델이 싸우는 방식이 아니라, **"노이즈를 예측하고 제거"**라는
> 명확한 목표를 갖기 때문에 학습이 안정적이다.

### SANA 모델이란?

> **💡 교안 복습 (슬라이드 34~35)**
>
> SANA는 NVIDIA NVlabs에서 개발한 경량 고해상도 이미지 생성 모델이다.
> 교안에서 배운 세 가지 기술이 합쳐져 있다:
> - **LDM (Latent Diffusion Model)**: 잠재 공간에서 작업하여 속도 향상 (슬라이드 19~20)
> - **CLIP**: 텍스트를 이해하여 이미지 생성을 안내 (슬라이드 24~25)
> - **지식 증류**: 거대 모델의 지식을 소형 모델에 전달하여 경량화 (슬라이드 31~32)
>
> 1.6B 파라미터의 비교적 작은 모델 크기로 1024×1024 고해상도 이미지를 빠르게 생성할 수 있다.

### HuggingFace Hub와 파이프라인

[**HuggingFace Hub**](https://huggingface.co/models)는 수만 개의 사전학습 모델을 호스팅하는 플랫폼이다.
`from_pretrained('모델ID')` 메서드로 모델을 불러올 수 있으며, 처음 실행 시 자동으로 모델을 다운로드하고 이후에는 로컬 캐시를 재사용한다.

> **💡 3-1 실습과의 연결**
>
> 3-1에서 ViT 모델을 `ViTForImageClassification.from_pretrained()`로 불러온 것과 동일한 방식이다.
> 이번에는 **이미지 분류가 아닌 이미지 생성** 모델을 불러온다.

`diffusers` 라이브러리의 `SanaPipeline`은 **텍스트 인코딩 → 노이즈 생성 → 반복 디노이징 → 디코딩**까지 전체 과정을 한 번의 호출로 처리하는 고수준 API이다.


### TODO 1: SANA 파이프라인 로드 및 이미지 생성

SANA 모델을 HuggingFace Hub에서 로드하고, 프롬프트를 사용하여 수채화 스타일의 여우 이미지를 생성한다.

**요구사항:**
1. `SanaPipeline.from_pretrained()`을 사용하여 `"Efficient-Large-Model/Sana_1600M_1024px_diffusers"` 모델을 로드
   - `torch_dtype=torch.float16`으로 설정
2. 로드한 파이프라인으로 이미지를 **2장** 생성
   - `positive_prompt`를 입력으로 사용
   - `guidance_scale`, `num_inference_steps`, `height`, `width` 파라미터를 적절히 설정
   - 각 이미지마다 **서로 다른 시드**의 `torch.Generator`를 사용

> **💡 프롬프트(Prompt)란?**
>
> 이미지 생성 모델에게 보내는 "주문서"이다. 좋은 프롬프트의 구성요소:
> - **주제**: "a red fox" (무엇을 그릴지)
> - **스타일**: "watercolor painting" (어떤 화풍으로)
> - **배경/상황**: "sitting on a forest floor" (어디에서)
> - **품질 키워드**: "vibrant autumn colors" (퀄리티 향상)

> **💡 주요 파라미터 설명**
>
> | 파라미터 | 의미 | 권장값 |
> |---------|------|:---:|
> | `guidance_scale` | 텍스트 조건을 얼마나 강하게 따를지 (높을수록 프롬프트에 충실) | 4.5 |
> | `num_inference_steps` | 노이즈 제거 반복 횟수 (많을수록 정교, 느림) | 20 |
> | `generator` | 시드 기반 난수 생성기 (같은 시드 = 같은 이미지) | - |

**힌트:**
- GPU 메모리가 부족할 경우 `pipe.to('cuda')` 대신 `pipe.enable_model_cpu_offload()`을 사용할 수 있다.
- float16에서 특정 시드 조합으로 검은 이미지가 나올 수 있다. 시드를 변경해 보자.


In [ ]:
from diffusers import SanaPipeline

# ========== 1. SANA 모델 로드 ==========
# HuggingFace Hub에 등록된 모델 ID
# 형식: "조직명/모델명" (3-1에서 ViT를 불러올 때와 동일한 방식)
model_id = 'Efficient-Large-Model/Sana_1600M_1024px_diffusers'

# from_pretrained: 모델 구조 + 학습된 가중치를 다운로드
# torch_dtype=torch.float16: 16비트로 로드하여 GPU 메모리 절약
# → 32비트(float32) 대비 메모리 절반 사용. 품질 차이는 거의 없음.
pipe = SanaPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)

# GPU로 모델 이동
# 왜 pipe.to(device)인가?
# → SanaPipeline 내부에는 텍스트 인코더, U-Net, VAE 등 여러 모델이 있다.
#   .to(device)를 하면 이 모든 컴포넌트가 한꺼번에 GPU로 이동한다.
#
# 메모리 부족 시 대안: pipe.enable_model_cpu_offload()
# → 컴포넌트를 하나씩만 GPU에 올려 메모리를 절약한다.
#   단, pipe.to('cuda')와 동시에 사용하면 안 된다.
pipe.to(device)
# pipe.enable_model_cpu_offload()  # 메모리 부족 시 이 줄을 활성화

# ========== 2. 이미지 생성 ==========
images = []
seeds = [42, 777]  # 서로 다른 시드 → 서로 다른 이미지 생성

# 프롬프트 설정
# 구성: [스타일] + [주제] + [배경] + [품질 키워드]
positive_prompt = 'A watercolor painting of a red fox sitting on a forest floor, vibrant autumn colors'

for i, seed in enumerate(seeds):
    # torch.Generator: 시드를 지정하여 재현 가능한 난수 생성기
    # 왜 시드마다 다른 이미지가 나오는가?
    # → 디퓨전 모델은 "순수 노이즈"에서 시작한다.
    #   시드가 다르면 시작 노이즈가 달라지므로 결과 이미지도 달라진다.
    generator = torch.Generator(device=device).manual_seed(seed)

    result = pipe(
        prompt=positive_prompt,     # 텍스트 프롬프트 (이것을 기반으로 이미지 생성)
        guidance_scale=4.5,         # 텍스트 준수 강도 (높을수록 프롬프트에 충실)
        num_inference_steps=20,     # 디노이징 반복 횟수 (많을수록 정교, 느림)
        height=1024,                # 생성 이미지 높이 (픽셀)
        width=1024,                 # 생성 이미지 너비 (픽셀)
        generator=generator,        # 시드별 난수 생성기
    )
    # result.images[0]: 생성된 PIL 이미지 (리스트의 첫 번째 요소)
    img = result.images[0]
    images.append(img)

# 생성된 이미지 저장 및 출력
for i, img in enumerate(images):
    img.save(f'generated_image_{i}.png')  # 다음 Step에서 사용하기 위해 저장
    display(img)
    print(f'이미지 {i} 저장 완료 (시드: {seeds[i]})')


## Step 2: CLIP 모델을 사용한 생성 이미지 평가

### CLIP(Contrastive Language-Image Pretraining)이란?

> **💡 교안 복습 (슬라이드 24~26)**
>
> CLIP은 OpenAI가 개발한 멀티모달 모델로, **이미지와 텍스트를 같은 임베딩 공간에 투영**하여 유사도를 계산한다.
> 4억 개의 이미지-텍스트 쌍으로 대조학습(Contrastive Learning)되어,
> 별도 학습 없이도 이미지가 어떤 텍스트 설명과 가장 어울리는지 판단할 수 있다.

**CLIP의 동작 원리 (3단계):**
1. **이미지 인코더**(ViT)가 이미지를 벡터로 변환
2. **텍스트 인코더**(Transformer)가 텍스트를 벡터로 변환
3. 두 벡터 간 **코사인 유사도**를 계산하여 매칭 점수 산출

> **💡 CLIP Score란? (슬라이드 26)**
>
> 생성된 이미지와 프롬프트 텍스트 간의 코사인 유사도를 CLIP으로 계산한 값이다.
> **점수가 높을수록 → 프롬프트에 맞는 이미지가 잘 생성된 것.**
> Midjourney, Stable Diffusion 등 모든 이미지 생성 서비스에서 품질 평가에 활용된다.

**이번 실습에서의 평가 방식:**
- 정답 레이블: `"a watercolor painting of a fox"` (수채화 여우 그림)
- 오답 레이블: 동물만 다른 것, 화풍만 다른 것, 스타일이 사진인 것
- CLIP이 정답 레이블에 가장 높은 점수를 주는지 확인한다


생성된 이미지를 CLIP 모델로 평가한다.

> **💡 왜 이런 방식으로 평가하는가?**
>
> 이미지 생성 모델에게 "수채화 여우"를 주문했는데, 실제로 수채화 여우가 잘 만들어졌는지 확인해야 한다.
> 사람이 눈으로 보면 바로 알 수 있지만, **자동으로 대량 평가**하려면 정량적 지표가 필요하다.
> CLIP은 "이 이미지가 수채화 여우인가, 유화 여우인가, 사진 여우인가"를 숫자로 비교해 준다.


### TODO 2: CLIP 이미지-텍스트 유사도 계산

CLIP 모델과 프로세서를 로드하고, 생성된 이미지와 텍스트 레이블 간 유사도를 계산한다.

**요구사항:**
1. `CLIPProcessor`와 `CLIPModel`을 `"openai/clip-vit-base-patch32"`에서 로드
2. `processor`를 사용하여 이미지와 텍스트 레이블을 전처리
   - `return_tensors='pt'`, `padding=True` 옵션 사용
3. CLIP 모델로 `logits_per_image`를 계산하고, `softmax`로 확률을 구한다

> **💡 CLIPProcessor의 역할**
>
> 3-1 실습에서 `ViTImageProcessor`가 이미지 전처리를 담당했던 것과 같은 역할이다.
> 차이점은 `CLIPProcessor`가 **이미지와 텍스트를 동시에** 전처리한다는 것이다.
> - 이미지: 224×224 리사이즈 + 정규화 → 텐서
> - 텍스트: 토크나이징 + 패딩 → 텐서

> **💡 logits_per_image란?**
>
> 이미지 1장과 텍스트 N개 사이의 유사도 점수이다.
> 예: 텍스트 4개라면 `[25.3, 18.7, 22.1, 15.9]` 형태의 벡터가 나온다.
> 이것을 `softmax`로 변환하면 합이 1인 확률 분포가 된다.


In [ ]:
# ========== 평가할 이미지 불러오기 ==========
# Step 1에서 저장한 이미지를 다시 불러온다
image = Image.open('generated_image_0.png')

# ========== 텍스트 레이블 정의 ==========
# 왜 여러 개의 레이블을 정의하는가?
# → CLIP은 "가장 유사한 텍스트를 찾는" 모델이다.
#   정답 1개와 오답 여러 개를 놓고, 모델이 정답을 제대로 찾는지 확인한다.
#   오답도 "비슷하지만 다른" 것으로 설정해야 모델의 세밀한 이해력을 확인할 수 있다.
labels = [
    'a watercolor painting of a fox',   # 정답: 수채화 + 여우
    'a watercolor painting of a dog',   # 오답 1: 수채화는 맞지만 동물이 다름
    'an oil painting of a fox',         # 오답 2: 여우는 맞지만 화풍이 다름
    'a photo of a fox'                  # 오답 3: 여우는 맞지만 그림이 아닌 사진
]

# ========== 1. CLIP 모델 로드 ==========
from transformers import CLIPProcessor, CLIPModel

clip_model_id = 'openai/clip-vit-base-patch32'

# CLIPProcessor: 이미지 + 텍스트를 동시에 전처리하는 도구
# CLIPModel: 이미지 인코더 + 텍스트 인코더가 하나로 합쳐진 모델
processor = CLIPProcessor.from_pretrained(clip_model_id)
clip_model = CLIPModel.from_pretrained(clip_model_id).to(device)

# ========== 2. 이미지 + 텍스트 전처리 ==========
# processor는 이미지와 텍스트를 동시에 받아서 모델 입력 형태로 변환한다
# return_tensors='pt': PyTorch 텐서로 반환
# padding=True: 텍스트 길이가 다를 수 있으므로 짧은 문장에 패딩 추가
inputs = processor(text=labels, images=image, return_tensors='pt', padding=True).to(device)

# ========== 3. 유사도 계산 ==========
with torch.no_grad():  # 추론만 하므로 기울기 계산 불필요
    outputs = clip_model(**inputs)
    # logits_per_image: 이미지 1장 vs 텍스트 4개의 유사도 점수
    # 예: tensor([[25.3, 18.7, 22.1, 15.9]])
    logits_per_image = outputs.logits_per_image

    # softmax: 유사도 점수를 확률 분포(합=1)로 변환
    # 왜 softmax를 쓰는가?
    # → 원래 점수는 절대값이라 비교가 어렵다.
    #   softmax로 변환하면 "이미지가 각 텍스트와 얼마나 어울리는지"를 %로 해석할 수 있다.
    probs = logits_per_image.softmax(dim=1)

# ========== 결과 출력 ==========
print('CLIP 유사도 점수 (raw):', logits_per_image.cpu().numpy())
print('CLIP 확률 분포 (softmax):', probs.cpu().numpy())

best_idx = logits_per_image.argmax(dim=1).item()
print(f'\nCLIP 예측 결과: \'{labels[best_idx]}\' 라벨이 가장 타당하다고 예측되었다.')
print(f'→ 정답({labels[0]})을 {"맞혔다 ✓" if best_idx == 0 else "틀렸다 ✗"}')


## Step 3: ResNet-50 기반 모델과의 결과 비교

### ResNet-50과 CLIP의 차이

> **💡 교안 복습 (슬라이드 5)**
>
> 동일한 이미지를 CNN(ResNet-50)과 멀티모달 모델(CLIP)로 분석하면,
> 두 모델의 근본적인 차이를 확인할 수 있다.

| 구분 | ResNet-50 (CNN) | CLIP (멀티모달) |
|------|:---:|:---:|
| **학습 데이터** | ImageNet 1000개 클래스 | 4억 이미지-텍스트 쌍 |
| **학습 방식** | 지도학습 (정답 레이블) | 대조학습 (이미지-텍스트 매칭) |
| **출력** | 고정된 1000개 클래스 확률 | 임의 텍스트와의 유사도 점수 |
| **화풍 인식** | ❌ 불가 (클래스에 없음) | ✅ 가능 (텍스트로 표현 가능) |
| **유연성** | 사전 정의된 범주만 분류 | 새로운 개념도 텍스트로 평가 가능 |

> **💡 이 Step에서 확인할 것**
>
> - ResNet-50: "red fox"라고만 출력 (여우 종류만 알 수 있음)
> - CLIP: "수채화 여우 그림 vs 유화 여우 그림 vs 사진 여우" 구분 가능
>
> **같은 이미지인데 모델에 따라 이해도가 완전히 다르다**는 것이 핵심이다.


### TODO 3: ResNet-50으로 이미지 분류

ImageNet으로 사전학습된 ResNet-50 모델을 로드하고, 생성된 이미지를 분류한다.

**요구사항:**
1. `models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)`로 모델 로드 후 `.eval()` 설정
2. 이미지를 ResNet-50 입력에 맞게 전처리:
   - 224×224 리사이즈 → ToTensor → ImageNet 정규화
3. `softmax`로 확률을 구하고 Top-5 결과를 출력

> **💡 3-1 실습과의 연결**
>
> 3-1에서는 ResNet-18을 사용했고, 여기서는 ResNet-50을 사용한다.
> 사용 방법은 동일하다: `.eval()` → 전처리 → `torch.no_grad()` 안에서 추론.
> 차이점은 3-1에서는 "학습"이 목적이었고, 여기서는 **"CLIP과의 비교"**가 목적이다.

> **💡 ImageNet 정규화 값은 어디서 나온 건가?**
>
> `mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`
> 이 값은 **ImageNet 전체 데이터셋**의 채널별 평균/표준편차이다.
> 3-1 실습에서 CIFAR-10의 평균/표준편차를 직접 계산한 것과 같은 원리인데,
> ImageNet은 워낙 많이 사용되므로 값이 이미 표준화되어 있다.


In [ ]:
# ========== 1. ResNet-50 모델 로드 ==========
# 왜 ResNet-50을 사용하는가?
# → CLIP과의 비교를 위해서이다. ResNet-50은 ImageNet 1000개 클래스만 학습했으므로
#   "여우"라는 것은 알지만 "수채화"인지 "유화"인지는 구분하지 못한다.
#   이 한계를 CLIP과 비교하여 확인하는 것이 이 Step의 목적이다.
resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2).to(device)
resnet50.eval()  # 평가 모드: Dropout, BatchNorm 비활성화

# ImageNet 클래스 이름 매핑 (1000개 클래스 이름 리스트)
imagenet_classes = models.ResNet50_Weights.IMAGENET1K_V2.meta['categories']
print(f'ImageNet 클래스 수: {len(imagenet_classes)}')
print(f'클래스 예시: {imagenet_classes[:5]}')

# ========== 2. 이미지 전처리 ==========
# 3-1 실습에서 한 것과 동일한 과정: Resize → ToTensor → Normalize
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),   # ResNet 입력 크기
    transforms.ToTensor(),            # PIL → Tensor + 0~255 → 0.0~1.0
    transforms.Normalize(             # ImageNet 표준 정규화 값
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# .unsqueeze(0): 배치 차원 추가
# 왜? → 모델은 [배치, 채널, 높이, 너비] 4차원 입력을 기대한다.
#        이미지 1장은 [채널, 높이, 너비] 3차원이므로 앞에 1을 추가한다.
img_tensor = preprocess(image).unsqueeze(0).to(device)

# ========== 3. 추론 + Top-5 결과 ==========
with torch.no_grad():  # 추론만 하므로 기울기 불필요
    output = resnet50(img_tensor)

# softmax로 확률 변환 (1000개 클래스에 대한 확률 분포)
probs = torch.nn.functional.softmax(output, dim=1)[0]

# Top-5 예측 결과
top5_prob, top5_idx = probs.topk(5)
top5_prob = top5_prob.cpu().numpy()
top5_idx = top5_idx.cpu().numpy()

class_idx = top5_idx[0]
class_name = imagenet_classes[class_idx]
print(f'\nResNet-50 예측 1순위: {class_name} (확률: {top5_prob[0]:.4f})')
print('Top-5 예측:')
for idx, prob in zip(top5_idx, top5_prob):
    print(f'  {imagenet_classes[idx]:20s} {prob:.4f}')
print('\n→ ResNet-50은 여우 종류(red fox, kit fox 등)만 출력한다.')
print('→ "수채화"인지 "유화"인지 "사진"인지는 구분하지 못한다.')


#### CLIP과 ResNet-50 비교 요약

위 결과를 비교하면:
- **CLIP**은 우리가 정의한 문장 단위의 라벨을 통해 이미지가 *"여우 그림"*임을 알아채고, 스타일(수채화/유화/사진)까지 구분할 수 있었다.
- **ResNet-50**은 해당 그림을 여우로 인식했지만, 결과는 여우 종류(red fox, kit fox 등)로만 출력될 뿐 화풍 정보는 담지 못한다.

> **💡 핵심 차이: 학습 데이터가 모델의 이해 범위를 결정한다**
>
> - ResNet-50은 ImageNet의 1000개 클래스(사물 이름)만 학습했으므로, **"무엇"**만 안다
> - CLIP은 4억 개의 이미지-텍스트 쌍을 학습했으므로, **"무엇 + 어떤 스타일 + 어떤 분위기"**까지 안다
>
> 이것이 **파운데이션 모델(CLIP)과 기존 CNN(ResNet)의 근본적 차이**이다.


## Step 4: 생성 데이터로 ResNet-18 전이학습 (Linear Probing)

### 전이학습과 Linear Probing (3-1 복습)

> **💡 3-1 실습에서 배운 내용**
>
> **전이학습**: 대규모 데이터로 사전학습된 모델의 특징 추출 능력을 새 태스크에 활용하는 기법
> **Linear Probing**: 전이학습의 가장 단순한 형태로, 모델 가중치를 전부 고정(Freeze)하고
> **마지막 fc층만 새로 학습**하는 방법
>
> ```python
> # 3-1에서 이미 했던 코드 패턴:
> for param in model.parameters():
>     param.requires_grad = False     # 동결
> model.fc = nn.Linear(512, 클래스수)  # fc층 교체
> optimizer = optim.SGD(model.fc.parameters(), lr=0.001)  # fc만 학습
> ```

### 합성 데이터(Synthetic Data)란?

> **💡 교안 복습 (슬라이드 36)**
>
> AI가 생성한 이미지를 **학습 데이터로 활용**하는 기법이다.
> 실제 데이터 수집이 어렵거나 개인정보 보호가 필요한 경우에 유용하다.

**이번 Step에서 하는 일:**
1. SANA로 "여우"와 "강아지" 이미지를 각각 생성 → **합성 데이터셋 구성**
2. 이 합성 데이터로 ResNet-18을 **Linear Probing** → 3-1에서 배운 전이학습의 **실전 응용**

즉, **Step 1에서 배운 이미지 생성 기술**과 **3-1에서 배운 전이학습 기술**이 이 Step에서 합쳐진다.


**전이학습에 사용할 학습/테스트 데이터를 SANA로 생성한다.**

각 클래스(fox, dog)별로 이미지를 생성하여 아래 폴더 구조로 저장한다.
이미지마다 서로 다른 시드를 사용하여 다양한 결과물을 확보한다.

```
data/
├── train/
│   ├── fox/    ← 여우 이미지 2장 (시드: 42, 777)
│   └── dog/    ← 강아지 이미지 2장 (시드: 42, 777)
└── test/
    ├── fox/    ← 여우 이미지 2장 (시드: 2024, 3000)
    └── dog/    ← 강아지 이미지 2장 (시드: 2024, 3000)
```

> **💡 왜 학습과 테스트 시드가 다른가?**
>
> 시드가 같으면 동일한 이미지가 생성된다. 학습 데이터와 테스트 데이터가 완전히 같으면
> "암기"한 것인지 "이해"한 것인지 구분할 수 없다.
> 서로 다른 시드로 생성하여 **처음 보는 이미지에 대한 진짜 실력**을 평가한다.

> **💡 ImageFolder란?**
>
> PyTorch에서 제공하는 유틸리티로, **폴더 이름을 클래스 레이블로 자동 인식**한다.
> `data/train/fox/` 폴더 안의 이미지는 자동으로 "fox" 레이블이 붙는다.
> 별도로 CSV 파일이나 레이블 매핑을 만들 필요가 없어 편리하다.


In [ ]:
# ========== 클래스별 프롬프트 정의 ==========
# 왜 프롬프트에 'detailed, vibrant'를 추가하는가?
# → 품질 키워드를 넣으면 더 선명하고 디테일한 이미지가 생성된다.
#   학습 데이터의 품질이 곧 모델 성능을 결정하므로 중요하다.
classes = {
    'fox': 'a watercolor painting of a red fox sitting on a forest floor, detailed, vibrant',
    'dog': 'a watercolor painting of a golden retriever sitting on grass, detailed, vibrant'
}

# 학습용과 테스트용으로 서로 다른 시드 사용 (데이터 겹침 방지)
train_seeds = [42, 777]
test_seeds = [2024, 3000]

# ========== 학습 데이터 생성 ==========
# os.makedirs: 폴더가 없으면 생성, exist_ok=True면 이미 있어도 에러 안 남
os.makedirs('data/train/fox', exist_ok=True)
os.makedirs('data/train/dog', exist_ok=True)

print('=== 학습 데이터 생성 ===')
for cls, prompt in classes.items():
    for i, seed in enumerate(train_seeds):
        generator = torch.Generator(device=device).manual_seed(seed)
        result = pipe(
            prompt=prompt,
            guidance_scale=4.5,
            num_inference_steps=20,
            height=1024,
            width=1024,
            generator=generator,
        )
        img = result.images[0]
        img.save(f'data/train/{cls}/{cls}_{i}.png')
        print(f'  [{cls}] 이미지 {i} 생성 완료 (시드: {seed})')
        display(img)

# ========== ImageFolder로 학습 데이터셋 구성 ==========
# 왜 RandomResizedCrop을 쓰는가?
# → 3-1에서 배운 데이터 증강(Data Augmentation)의 적용이다.
#   이미지 일부를 무작위로 잘라 224×224로 리사이즈 → 위치 변화에 강인하게 학습
# 왜 RandomHorizontalFlip을 쓰는가?
# → 50% 확률로 좌우 반전 → 좌우 대칭성 학습 (3-1에서 배운 기법)
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop((224, 224)),  # 무작위 자르기 + 리사이즈 (증강)
    transforms.RandomHorizontalFlip(),          # 좌우 반전 (증강)
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # ImageNet 정규화
])

# ImageFolder: 폴더 이름(fox, dog)을 자동으로 클래스 레이블로 인식
train_dataset = ImageFolder('data/train', transform=train_transforms)
train_loader = DataLoader(train_dataset, batch_size=5, shuffle=True)
print(f'\n학습 이미지 수: {len(train_dataset)} (클래스: {train_dataset.classes})')

# ========== 테스트 데이터 생성 ==========
os.makedirs('data/test/fox', exist_ok=True)
os.makedirs('data/test/dog', exist_ok=True)

print('\n=== 테스트 데이터 생성 ===')
for cls, prompt in classes.items():
    for i, seed in enumerate(test_seeds):
        generator = torch.Generator(device=device).manual_seed(seed)
        result = pipe(
            prompt=prompt,
            guidance_scale=4.5,
            num_inference_steps=20,
            height=1024,
            width=1024,
            generator=generator,
        )
        img = result.images[0]
        img.save(f'data/test/{cls}/{cls}_{i}.png')
        print(f'  [{cls}] 이미지 {i} 생성 완료 (시드: {seed})')
        display(img)

# 테스트 전처리: 증강 없이 단순 리사이즈만 적용 (공정한 평가를 위해)
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
test_dataset = ImageFolder('data/test', transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=5, shuffle=True)
print(f'\n테스트 이미지 수: {len(test_dataset)} (클래스: {test_dataset.classes})')


### TODO 4: ResNet-18 전이학습 구현 (Linear Probing)

사전학습된 ResNet-18 모델을 로드하고, 3-1 실습에서 배운 것과 동일한 방법으로
기존 가중치를 고정한 뒤 출력층을 교체하여 합성 데이터로 학습한다.

**요구사항:**
1. `models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)`으로 모델 로드
2. 기존 가중치 고정 (Freeze): `requires_grad = False`
3. 출력층 교체: `model.fc`를 우리 클래스 수(2개: fox, dog)에 맞게 교체
4. 손실 함수(`nn.CrossEntropyLoss`)와 옵티마이저(`optim.SGD`, lr=0.001, momentum=0.9) 정의
5. 3 에포크 동안 학습 루프 작성

> **💡 3-1 실습과 완전히 동일한 패턴**
>
> | 항목 | 3-1 실습 | 이번 실습 |
> |------|:---:|:---:|
> | 모델 | ResNet-18 | ResNet-18 (동일!) |
> | 데이터 | CIFAR-10 (실제 데이터) | **SANA로 생성한 합성 데이터** |
> | 클래스 수 | 10개 | **2개** (fox, dog) |
> | 기법 | Linear Probing | Linear Probing (동일!) |
>
> 코드 패턴은 3-1과 동일하되, **데이터만 "AI가 생성한 이미지"로 바뀐 것**이다.
> 이것이 합성 데이터의 핵심: **실제 데이터 없이도 모델을 학습시킬 수 있다.**


In [ ]:
# ========== 1. ResNet-18 모델 로드 ==========
# 3-1 실습에서 사용한 것과 동일한 모델
# pretrained=True 대신 weights= 사용 (최신 PyTorch 권장 방식)
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# ========== 2. 가중치 동결 ==========
# 3-1에서 배운 것과 동일: 특징 추출부는 ImageNet 지식을 그대로 유지
# requires_grad=False → 역전파 시 이 파라미터의 기울기를 계산하지 않음 → 값 불변
for param in model.parameters():
    param.requires_grad = False

# ========== 3. 출력층 교체 ==========
# 3-1: model.fc = nn.Linear(512, 10)  # CIFAR-10 → 10개 클래스
# 이번: model.fc = nn.Linear(512, 2)  # fox/dog → 2개 클래스
# in_features를 자동으로 가져오면 모델을 바꿔도 코드 수정 불필요
num_features = model.fc.in_features  # 512
model.fc = nn.Linear(num_features, len(train_dataset.classes))  # 512 → 2

model = model.to(device)

# 확인: 학습 가능 파라미터 수
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'학습 가능 파라미터: {trainable:,} / 전체: {total:,} ({trainable/total*100:.1f}%)')

# ========== 4. 손실 함수 + 옵티마이저 ==========
# CrossEntropyLoss: 분류 문제의 표준 손실 함수 (3-1과 동일)
criterion = nn.CrossEntropyLoss()

# 왜 model.fc.parameters()만 전달하는가?
# → Linear Probing이므로 fc층만 학습한다. 나머지는 동결된 상태.
# momentum=0.9: SGD에 관성을 추가하여 학습을 안정화 (3-1에서 배운 개념)
optimizer = optim.SGD(model.fc.parameters(), lr=0.001, momentum=0.9)

# ========== 5. 학습 루프 ==========
# 3-1에서 배운 5단계 학습 패턴과 완전히 동일
model.train()  # 학습 모드
num_epochs = 3

for epoch in range(num_epochs):
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()       # ① 기울기 초기화
        outputs = model(inputs)     # ② 순전파
        loss = criterion(outputs, labels)  # ③ 손실 계산
        loss.backward()             # ④ 역전파
        optimizer.step()            # ⑤ 가중치 업데이트

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(train_dataset)
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}')


학습이 완료되었다. 테스트 데이터(학습 때 사용하지 않은 이미지)로 모델의 진짜 실력을 평가한다.

> **💡 관찰 포인트**
>
> - 합성 데이터 4장(학습 2장 + 테스트 2장)만으로도 분류가 되는가?
> - 3-1에서 CIFAR-10 50,000장으로 학습한 것과 비교하면 데이터가 극도로 적다.
>   그런데도 어느 정도 성능이 나온다면, **사전학습된 ResNet의 특징 추출 능력이 그만큼 강력하다**는 의미이다.


In [ ]:
# ========== 역정규화 함수 ==========
# 왜 역정규화가 필요한가?
# → 모델 입력을 위해 Normalize로 평균 0, 표준편차 1 근처로 변환했다.
#   이 상태로 이미지를 표시하면 색상이 왜곡된다.
#   역정규화를 통해 원래 [0, 1] 범위로 복원해야 올바른 색상으로 표시된다.
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

def denormalize(img_tensor, mean=mean, std=std):
    """정규화된 이미지 텐서를 [0, 1] 범위로 복원한다."""
    # view(-1, 1, 1): [3] → [3, 1, 1]으로 변환하여 브로드캐스팅 가능하게
    mean_t = torch.tensor(mean, device=img_tensor.device).view(-1, 1, 1)
    std_t = torch.tensor(std, device=img_tensor.device).view(-1, 1, 1)
    # 정규화 역연산: x_원본 = x_정규화 * std + mean
    img = img_tensor * std_t + mean_t
    return img.clamp(0, 1)  # 0~1 범위 벗어나는 값 잘라내기

# ========== 모델 평가 ==========
model.eval()  # 평가 모드: Dropout/BatchNorm 비활성화 (3-1에서 배운 내용)
correct = 0
total = 0

with torch.no_grad():  # 추론만 하므로 기울기 계산 불필요 → 메모리 절약
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        predicted = outputs.argmax(dim=1)  # 가장 높은 점수의 클래스 = 예측

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # 예측 결과 시각화
        for img, pred in zip(inputs, predicted):
            _img = denormalize(img).cpu()
            print(f'예측 결과: {train_dataset.classes[pred]}')
            display(transforms.ToPILImage()(_img))

print(f'\n테스트 정확도: {100 * correct / total:.2f}%')
print(f'→ 합성 데이터 {len(train_dataset)}장만으로 Linear Probing을 수행한 결과이다.')


## 자가 체크리스트

아래 항목을 스스로 점검한다. 설명할 수 있으면 ✅, 불확실하면 해당 Step을 다시 복습한다.

### 개념 이해
- [ ] 디퓨전 모델이 이미지를 생성하는 원리(노이즈 제거)를 설명할 수 있다
- [ ] CLIP이 이미지와 텍스트를 비교하는 방식(코사인 유사도)을 설명할 수 있다
- [ ] CLIP Score가 무엇인지, 왜 이미지 생성 품질 평가에 사용되는지 설명할 수 있다
- [ ] ResNet(CNN)과 CLIP(멀티모달)의 근본적 차이를 설명할 수 있다
- [ ] 합성 데이터가 무엇이고, 왜 필요한지 설명할 수 있다

### 코드 구현
- [ ] HuggingFace에서 SANA 파이프라인을 로드하고 이미지를 생성할 수 있다
- [ ] CLIP 모델로 이미지-텍스트 유사도를 계산할 수 있다
- [ ] ResNet-18의 가중치를 고정하고 fc층을 교체하여 Linear Probing을 구현할 수 있다

### 결과 해석
- [ ] CLIP과 ResNet-50의 분류 결과 차이를 해석할 수 있다
- [ ] 합성 데이터로 학습한 모델의 성능을 평가하고 한계를 설명할 수 있다

### 심화 도전 (선택)
- [ ] 프롬프트를 변경하여(유화, 수묵화, 픽셀아트 등) CLIP Score 변화를 분석해 본다
- [ ] fox/dog 2클래스 대신 3~5개 동물 클래스로 합성 데이터셋을 확장해 본다
- [ ] `guidance_scale`이나 `num_inference_steps`를 바꿔보며 생성 품질 차이를 관찰한다

### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 삼성청년SW·AI아카데미의 컨텐츠 자산으로, 보안서약서에 의거하여 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.
